## ?? 6

**Kaggle ?** train_prob.csv? ?? target? ???? ??? ???, 

test_prob.csv? ?? target ???? ??? ?? ??? answer6.csv? ????.

id, target

0, 6.9

5, 7.8

...


**????**

$RMSE(Y, \hat{Y}) = \sqrt{\frac{1}{n}\sum^{n}_{i=1}(y_i-\hat{y_i})^2}$

**??: ????? ???(sunku0316.kang@multicampus.com, sun9sun9@gmail.com)**

In [ ]:
# ?? ?? ??

import pandas as pd
import numpy as np
import sklearn
import scipy
import statsmodels
import mlxtend
import sys
import xgboost as xgb

print(sys.version)
for i in [pd, np, sklearn, scipy, mlxtend, statsmodels, xgb]:
    print(i.__name__, i.__version__)

In [ ]:
df_train = pd.read_csv('train_prob.csv', index_col='id')
df_test = pd.read_csv('test_prob.csv', index_col='id')
df_ans = pd.read_csv('test_prob_ans.csv', index_col='id')

In [ ]:
# ?? ??? ??? ???? list ??? ?????.
repl_list = [
    ('cat3', {'B': 'C'}, [83634, 147361, 9005]),
    ('cat4', {'A': 'B', 'D': 'B'}, [239397, 603]),
    ('cat6', {'D': 'A', 'E': 'B', 'G': 'C', 'H': 'B', 'I': 'A'}, [234203, 5145, 652]),
    ('cat7', {'A': 'B', 'C': 'B', 'F': 'D', 'I': 'B'}, [4606, 19784, 214027, 1583]),
    ('cat8', {'B': 'G', 'F': 'E'}, [30338, 96743, 2953, 76085, 33881]),
    ('cat9', {'C': 'H', 'D': 'B', 'E': 'L'}, [10678, 2846, 85944, 8320, 19987, 40070, 5501, 16743, 33793, 7819, 3331, 4968])
]
# ??? ?? ???? ?????.
for v, d, cnts in repl_list:
    print(v, d, cnts)
    # ??? ??? s_repl? ?????
    s_repl = df_train[v].replace(d)
    # ????? ?????.
    if (s_repl.nunique() != len(cnts)) or ((s_repl.value_counts().sort_index() != cnts).any()):
        print(s_repl.value_counts())
        break
    df_train[v] = s_repl
    df_test[v] = df_test[v].replace(d)
    
cat_cols = ['cat{}'.format(i) for i in range(10)]
cont_cols = ['cont{}'.format(i) for i in range(14)]

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from scipy.stats import norm

import os

X_all = df_test.columns.tolist()
df_train['targetA'] = df_train['target'] <= 7.45

# ??? ?????? ?????, ?? ??? ?? ? ?????.
# ?? ??? ??? ???? ???, ? ??? ??? ? ? ? ??? ??? ? ?? ??? ????.
# ????? ? ????? ???  targetA_train.csv? ???? ???.
if not os.path.isfile('targetA_train.csv'):
    # ?? 3?? targetA? ?? ?? ??? ?????,
    df_train_clf = df_train.assign(
        prob_A = 1 - norm.cdf(df_train['target'], loc=6.769, scale=0.616),
        prob_B = norm.cdf(df_train['target'], loc=8.123, scale=0.527)
    ).query('prob_B < 0.01 or prob_A < 0.01').copy()
    
    clf_xgb = make_pipeline( 
        ColumnTransformer([
            ('ohe', OneHotEncoder(handle_unknown='ignore'), cat_cols),
            ('pt', 'passthrough', cont_cols)
        ]),
        xgb.XGBClassifier(
            max_depth = 2, # ??? ?? ?? 2
            reg_alpha = 0.1, # L1 ?? 0.1
            reg_lambda = 0.1, # L2 ?? 0.1
            colsample_bytree=0.25, # ?? ? ?? ??? ?? 0.25
            n_estimators=500, # ??? ? 500
            random_state=123, # random_state 123
        )
    )
    
    clf_xgb.fit(df_train_clf[X_all], df_train_clf['targetA'])
    df_targetA_train = pd.DataFrame({'targetA_prob': clf_xgb.predict_proba(df_train[X_all])[:, 1]}, index=df_train.index)
    # ???? ??? ???? ?? ?? ??? ?? ? ????.
    
    # ? ?? ??? ??? ?? ???? ??? ???.
    # cross_val_predict? ?? ? ? ????. ?? ??? ???? ?? ??? ??? ?????.
    # train?? ?? ?? ??? ?? train? ??? ??? ?? ??? ???? ???.
    targetA_prob_cv = cross_val_predict(clf_xgb, df_train_clf[X_all], df_train_clf['targetA'], cv=5, method='predict_proba')
    # df_train_clf? ?? ???? ??? ????? ????? ?????.
    # ? ??? ?? ?? ? ?? ?? ?? ??? ???????, ? ??? ???? ? ????.
    df_targetA_train.loc[df_targetA_train.index.isin(df_train_clf.index), 'targetA_prob'] = targetA_prob_cv[:, 1]
    
    df_targetA_train.to_csv('targetA_train.csv')
    df_targetA_test = pd.DataFrame({'targetA_prob': clf_xgb.predict_proba(df_test[X_all])[:, 1]}, index=df_test.index)
    df_targetA_test.to_csv('targetA_test.csv')
else:
    # ??? ??? ?????.
    df_targetA_train = pd.read_csv('targetA_train.csv', index_col='id')
    df_targetA_test = pd.read_csv('targetA_test.csv', index_col='id')

In [ ]:
df_train['targetA_prob'] = df_targetA_train['targetA_prob']
df_test['targetA_prob'] = df_targetA_test['targetA_prob']

In [ ]:
q = [i for i in np.arange(0, 1.01, 0.01)]
# ??? ??? ???? ?? ?? ??? ??? ???.
for i in cont_cols:
    q_val = df_train[i].quantile(q)
    q_val.iloc[[0, -1]] = [-np.inf, np.inf]
    q_cut = pd.cut(df_train[i], bins=q_val)
    q_mean = df_train.groupby(q_cut)['target'].mean()
    df_train[i + '_q'] = q_cut.map(q_mean).astype('float')
    df_test[i + '_q'] = pd.cut(df_test[i], bins=q_val).map(q_mean).astype('float')

In [ ]:
s_hist = []

In [ ]:
# ??
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import ShuffleSplit, KFold

cv = KFold(n_splits=5, random_state=123)
# train(80%)/test(20%) ? ??? ?????. XGB, RF? ?? ??? ??? ?? ?????.
ss = ShuffleSplit(n_splits=1, train_size=0.8, random_state=123)

df_ans = pd.read_csv('test_prob_ans.csv', index_col='id')
X_all = df_test.columns.tolist()  + ['targetA_prob']

cat_cols = ['cat{}'.format(i) for i in range(10)]
cont_cols = ['cont{}'.format(i) for i in range(14)]
cont_q_cols = ['cont{}_q'.format(i) for i in range(14)]

# ??? ??? leak? ?? ?? ????? ???.
q = [i for i in np.arange(0, 1.01, 0.01)]
def eval_model(model_name, model, sp):
    score_train, score_valid = list(), list()
    for train_idx, valid_idx in sp.split(df_train):
        df_cv_train, df_valid = df_train.iloc[train_idx].copy(), df_train.iloc[valid_idx].copy()
        # ???? ???? ??? ? ???, ?? ?? leakage? ?? ??? ??? ??? ??? ???? cross_validate? 
        # ???? ??? ?? ???.
        # ????? train?? ????? ???
        # ???? test(???)? ???? train?? ?? ???(mean)? ?????.
        for i in cont_cols:
            q_val =  df_cv_train[i].quantile(q)
            q_val.iloc[[0, -1]] = [-np.inf, np.inf]
            q_cut = pd.cut(df_cv_train[i], bins=q_val)
            q_mean = df_cv_train.groupby(q_cut)['target'].mean()
            df_cv_train[i + '_q'] = q_cut.map(q_mean).astype('float')
            df_valid[i + '_q'] = pd.cut(df_valid[i], bins=q_val).map(q_mean).astype('float')
        model.fit(df_cv_train[X_all], df_cv_train['target'])
        score_valid.append((mean_squared_error(df_valid['target'], model.predict(df_valid[X_all]))) ** 0.5)
        score_train.append((mean_squared_error(df_cv_train['target'], model.predict(df_cv_train[X_all]))) ** 0.5)
        
    output = 'Valid: {:.5f}?{:.5f},  V.Train: {:.5f}?{:.5f}'.format( 
            np.mean(score_valid), np.std(score_valid),
            np.mean(score_train), np.std(score_train),
        )
    print(output)
    s_hist.append(pd.Series([model_name, output, np.mean(score_valid)], index=['model name', 'result', 'score']))
    df_result = pd.DataFrame(s_hist)
    display(df_result.groupby('model name').last())
    display(df_result.loc[df_result['score'].idxmin()])

def select_model(model):
    model.fit(df_train[X_all], df_train['target'])
    prd = model.predict(df_test[X_all])
    pd.DataFrame({
        'id': df_test.index.values,
        'target': prd
    }).to_csv('answer6.csv', index = None)
    return prd 

In [ ]:
from sklearn.linear_model import LinearRegression
# ???? ?? ?????.
ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), cat_cols),
    ('std', StandardScaler(), cont_cols)
])

reg_lr = make_pipeline(ct, LinearRegression())
eval_model('baseline', reg_lr, cv)

In [ ]:
prd = select_model(reg_lr)
print("baseline ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)

In [ ]:
# + targetA_prob ? ?????.
ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), cat_cols),
    ('std', StandardScaler(), cont_cols),
    ('pt', 'passthrough', ['targetA_prob'])
])

reg_lr2 = make_pipeline(ct, LinearRegression())
eval_model('lr2', reg_lr2, cv)

In [ ]:
prd = select_model(reg_lr2)
print("lr2 ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)

In [ ]:
# + targetA_prob ? ?????.
# + cont{}_q ? cont{} ??? ?????.

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), cat_cols),
    ('std', StandardScaler(), cont_q_cols),
    ('pt', 'passthrough', ['targetA_prob'])
])

reg_lr3 = make_pipeline(ct, LinearRegression())
eval_model('lr3', reg_lr3, cv)

In [ ]:
prd = select_model(reg_lr3)
print("lr3 ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)

In [ ]:
# + targetA_prob ? ?????.
# PCA + cont{}_q ? cont{} ??? ?????.

from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), cat_cols),
    ('std', make_pipeline(StandardScaler(), PCA(n_components=0.95)), cont_q_cols),
    ('pt', 'passthrough', ['targetA_prob'])
])
reg_lr4 = make_pipeline(
    ct, 
    LinearRegression()
)
eval_model('lr4', reg_lr4, cv)

In [ ]:
prd = select_model(reg_lr4)
print("lr4 ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(), cat_cols),
    ('pt', 'passthrough', cont_cols + ['targetA_prob'])
])

reg_rf = make_pipeline(
    ct,
    RandomForestRegressor(n_estimators=50, max_depth=4, random_state=123, n_jobs=4) # Kaggle??? ??????? ?? ???.
)
# RandomForestRegressor? ??? ??????. 
# ?? ??? ?????? Hold Out (Shuffle Split n_splits=1) ???? ????.
eval_model('rf', reg_rf, ss)

In [ ]:
ct = ColumnTransformer([
    ('ohe', OneHotEncoder(), cat_cols),
    ('pt', 'passthrough', cont_cols + ['targetA_prob'])
])
reg_xgb = make_pipeline(
    ct,
    xgb.XGBRegressor(n_estimators=150, max_depth=2, random_state=123, n_jobs=4)
)
# Hold Out (Shuffle Split n_splits=1) ??? ?????.
eval_model('xgb', reg_xgb, ss)

In [ ]:
# ? ? ??? ????.
prd = select_model(reg_xgb)
print("xgb ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)

In [ ]:
# Voting? ?? ???? ????.
from sklearn.ensemble import VotingRegressor

reg_vt = VotingRegressor([
    #('baseline', reg_lr),
    #('lr2', reg_lr2),
    ('lr3', reg_lr3),
    ('lr4', reg_lr4),
    ('rf', reg_rf),
    ('xgb', reg_xgb),
])
eval_model('vt', reg_vt, ss)

In [ ]:
# ??? ???? ??? ?????. 
prd = select_model(reg_vt)
# ?? ??? ????.
print("Voting ?? ??:",  mean_squared_error(df_ans['target'], prd) ** 0.5)